In [ ]:
import torch
import torchvision.transforms as transforms
import cv2
import os
from PIL import Image
import numpy as np
from datetime import datetime
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes=2):
    """Initialize the model"""
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def load_image(image_path):
    """Load and preprocess image"""
    image = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    return transform(image), image

def create_output_dirs(base_path):
    """Create output directories for annotated images and text files"""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    annotated_dir = os.path.join(base_path, f'annotated_images_{timestamp}')
    annotation_dir = os.path.join(base_path, f'annotations_{timestamp}')

    os.makedirs(annotated_dir, exist_ok=True)
    os.makedirs(annotation_dir, exist_ok=True)

    return annotated_dir, annotation_dir

def draw_boxes_and_save(original_image, boxes, scores, threshold, output_path):
    """Draw bounding boxes on image and save"""
    image_np = np.array(original_image)
    image_np = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    # Draw each box that meets the confidence threshold
    for box, score in zip(boxes, scores):
        if score > threshold:
            x1, y1, x2, y2 = map(int, box)

            # Draw rectangle
            cv2.rectangle(image_np, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Add confidence score
            text = f"{score:.2f}"
            cv2.putText(image_np, text, (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imwrite(output_path, image_np)

def save_annotations(boxes, scores, threshold, image_name, output_path):
    """Save detection results to text file"""
    with open(output_path, 'w') as f:
        f.write(f"Detection results for {image_name}\n")
        f.write("Format: <confidence> <x1> <y1> <x2> <y2>\n\n")

        for box, score in zip(boxes, scores):
            if score > threshold:
                x1, y1, x2, y2 = map(int, box)
                f.write(f"{score:.3f} {x1} {y1} {x2} {y2}\n")

def run_inference(model_path, image_dir, output_base_dir, conf_threshold=0.5):
    """Run inference on all images in directory"""
    # Set up device and model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_model()

    # Load trained weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # Create output directories
    annotated_dir, annotation_dir = create_output_dirs(output_base_dir)
    print(f"Saving annotated images to: {annotated_dir}")
    print(f"Saving text annotations to: {annotation_dir}")

    # Process each image
    total_images = len([f for f in os.listdir(image_dir)
                       if f.endswith(('.png', '.jpg', '.tif'))])
    processed = 0

    for image_name in os.listdir(image_dir):
        if not image_name.endswith(('.png', '.jpg', '.tif')):
            continue

        image_path = os.path.join(image_dir, image_name)

        try:
            # Load and preprocess image
            image_tensor, original_image = load_image(image_path)

            # Run inference
            with torch.no_grad():
                predictions = model([image_tensor.to(device)])

            # Get predictions
            pred_boxes = predictions[0]['boxes'].cpu().numpy()
            pred_scores = predictions[0]['scores'].cpu().numpy()

            # Save annotated image
            output_image_path = os.path.join(
                annotated_dir,
                f'annotated_{os.path.splitext(image_name)[0]}.png'
            )
            draw_boxes_and_save(original_image, pred_boxes, pred_scores,
                              conf_threshold, output_image_path)

            # Save text annotations
            output_text_path = os.path.join(
                annotation_dir,
                f'detection_{os.path.splitext(image_name)[0]}.txt'
            )
            save_annotations(pred_boxes, pred_scores, conf_threshold,
                           image_name, output_text_path)

            processed += 1
            print(f"Processed {processed}/{total_images}: {image_name}")

        except Exception as e:
            print(f"Error processing {image_name}: {str(e)}")
            continue

    print("\nInference completed!")
    print(f"Processed {processed} images")
    print(f"Annotated images saved to: {annotated_dir}")
    print(f"Text annotations saved to: {annotation_dir}")

if __name__ == "__main__":
    # Configuration
    model_path = 'best_model.pth'  # Path to your trained model
    image_dir = '/content/drive/MyDrive/SPIE_Paper/synthetic'  # Directory with images
    output_base_dir = '/content/drive/MyDrive/SPIE_Paper/Results'  # Base directory for results
    confidence_threshold = 0.5  # Confidence threshold for detections

    # Run inference
    run_inference(model_path, image_dir, output_base_dir, confidence_threshold)